All thresholds come from known validation scores only. Ties are accepted.

In [ ]:
def calibrate_threshold(u):
    u=np.asarray(u,dtype=np.float64)
    if u.ndim!=1 or not np.isfinite(u).all():raise ValueError('Invalid calibration scores.')
    return float(np.quantile(u,.95,method='linear'))

def source_fingerprints():
    result={}
    for p in sorted(ROOT.rglob('*.ipynb')):
        if any(part.startswith('.') for part in p.relative_to(ROOT).parts):continue
        obj=read_json(p)
        code=[''.join(c['source']) for c in obj['cells'] if c['cell_type']=='code' and 'parameters' not in c.get('metadata',{}).get('tags',[])]
        result[str(p.relative_to(ROOT))]=hashlib.sha256(json.dumps(code).encode()).hexdigest()
    for p in sorted((ROOT/'configs').glob('*.yaml')):result[str(p.relative_to(ROOT))]=sha(p)
    return result

def verify_lock():
    p=ROOT/'results/protocol_lock.json'
    if not p.exists():raise RuntimeError('Run known extraction, then calibration before any unknown evaluation.')
    lock=read_json(p)
    if lock['sources']!=source_fingerprints():raise RuntimeError('Code/config changed after calibration; keep this experiment fixed and use a new folder for changes.')
    for method,digest in lock['checkpoints'].items():
        if sha(ROOT/'results'/method/'best.pt')!=digest:raise RuntimeError('Checkpoint changed after calibration.')
    for relative,digest in lock['artifacts'].items():
        if sha(ROOT/relative)!=digest:raise RuntimeError('Known cache or calibration artifact changed.')
    return lock